# M5 · Validación del PR #22 y prueba de caída (María Alejandra)

Corre el harness (`m4/harness_m4.py`) con la configuración de la Demo (**fp16**) y guarda todo en `m5/`, sin tocar los resultados históricos de `m4/`.

**Antes de correr, en Kaggle (panel derecho → Settings):**
- *Accelerator* → **GPU T4 x1**
- *Internet* → **On** (la cuenta tiene que estar verificada con teléfono)
- La rama `maria-m5` tiene que estar subida a GitHub con el harness nuevo.

## 1 · Entorno: clonar la rama e instalar

In [1]:
import os
RAMA = "maria-m5"
if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia
!git log -1 --oneline
!grep -c "TODAS_LAS_CORRIDAS" m4/harness_m4.py   # debe ser mayor que 0: es el harness nuevo

Cloning into 'triage-dermatologico-ia'...
remote: Enumerating objects: 723, done.
remote: Counting objects: 100% (298/298), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 723 (delta 173), reused 211 (delta 138), pack-reused 425 (from 1)
Receiving objects: 100% (723/723), 17.22 MiB | 31.77 MiB/s, done.
Resolving deltas: 100% (355/355), done.
/kaggle/working/triage-dermatologico-ia
71b64dc (HEAD -> maria-m5, origin/maria-m5) M5: harness con --salida, sin_senal y prueba de caida + notebook de Kaggle
5


In [2]:
%pip install -q -r requirements.txt
# Mismo arreglo que en los notebooks de M2-M5: torchao viejo hace tronar a peft al importarse.
!pip uninstall -y -q torchao

import torch, transformers
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
GPU disponible: True
GPU: Tesla T4
torch 2.11.0+cu128 | transformers 5.16.1


## 2 · Prueba de humo (2 casos por corrida, ~5 min)
Solo confirma que todo carga. Va a `m5/humo/`, que no se sube al repo.

In [3]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/humo --limite 2

config.json: 100%|█████████████████████████████| 661/661 [00:00<00:00, 3.68MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 25.9MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 50.8MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 126MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 153MB/s]
model.safetensors.index.json: 100%|████████| 35.6k/35.6k [00:00<00:00, 85.1MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 6.17GB            
Reconstructing (incomplete total...):   5%|▍       |  292MB / 6.17GB,  109MB/s  
Reconstructing (incomplete total...):  64%|█████   | 3.95GB / 6.17GB,  877MB/s  
Reconstructing (incomplete total...):  70%|█████▋  | 4.35GB / 6.17GB,  125MB/s  

Fetching 2 files:  50%|███

## 3 · Corrida completa: validación del PR #22 (~15-20 min)
con_foto (10) + sin_foto (10) + m3_texto (37). La latencia pareada debe salir con n=9.

In [4]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/harness

Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 2616.76it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 8289.14it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.Laye

## 4 · Prueba de caída: BETO y RAG apagados
Los 10 casos sin foto. Todos deben salir `sin_senal`, ninguno `no_urgente`, y la respuesta debe decir «revisión manual» → `"pasa": true`.

In [7]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/caida --corridas caida

Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 2704.56it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█████████████████████| 197/197 [00:00<00:00, 11679.17it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dens

## 5 · Resultados

In [8]:
import json
r = json.load(open("m5/harness/resultados.json", encoding="utf-8"))
print("ENTORNO:", json.dumps(r.get("entorno"), ensure_ascii=False, indent=1))
for n, c in r["corridas"].items():
    print(f"\n{n}: recall={c['d3_recall_urgente']}  D1={c['d1_aciertos']}  sin_senal={c['casos_sin_senal']}  "
          f"p50={c['latencia_s_p50']} s  p95={c['latencia_s_p95']} s  rutas={c['rutas_cierre']}")
print("\nLatencia pareada:", r.get("latencia_pareada_foto"))
print("\n" + open("m5/harness/tabla_delta.md", encoding="utf-8").read())
caida = json.load(open("m5/caida/resultados.json", encoding="utf-8"))["prueba_caida"]
print("PRUEBA DE CAÍDA:", json.dumps(caida, ensure_ascii=False, indent=1))

ENTORNO: {
 "fecha_utc": "2026-11-02T15:12:30Z",
 "corridas": [
  "con_foto",
  "sin_foto",
  "m3_texto"
 ],
 "limite": null,
 "TRIAGE_CUANT": "fp16",
 "git_commit": "71b64dc",
 "torch": "2.11.0+cu128",
 "transformers": "5.16.1",
 "gpu": "Tesla T4",
 "dtype_cerebro": "float16",
 "dtype_generador": "float16"
}

con_foto: recall=0.8  D1=4/8  sin_senal=0  p50=14.46 s  p95=15.84 s  rutas={'sintesis_por_repeticion': 7, 'sintesis_pasos_agotados': 3}

sin_foto: recall=0.8  D1=4/8  sin_senal=0  p50=11.75 s  p95=13.28 s  rutas={'sintesis_por_repeticion': 10}

m3_texto: recall=0.7  D1=23/32  sin_senal=1  p50=12.1 s  p95=16.66 s  rutas={'sintesis_por_repeticion': 28, 'responder_final': 9}

Latencia pareada: {'n': 9, 'con_foto_media': 14.56, 'sin_foto_media': 11.79, 'delta_media_s': 2.78, 'delta_mediana_s': 3.01, 'delta_pct': 23.6}

| Medida | M3 (sin foto) | Sistema actual (con foto) |
|---|---|---|
| Casos sin imagen (37 de M3): D1 | 23/32 | 23/32 |
| Casos sin imagen: recall urgentes | 0.7 | 0.

## 6 · Empaquetar
Descarga `m5_harness.zip` desde *Output* (panel derecho) y extráelo en la raíz del repo: sus rutas empiezan por `m5/`.

In [9]:
import os, zipfile
with zipfile.ZipFile("/kaggle/working/m5_harness.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["m5/harness", "m5/caida"]:
        for d, _, fs in os.walk(base):
            for f in fs:
                z.write(os.path.join(d, f))
print(zipfile.ZipFile("/kaggle/working/m5_harness.zip").namelist())

['m5/harness/tabla_delta.md', 'm5/harness/scorecard_con_foto.csv', 'm5/harness/entorno.json', 'm5/harness/resultados.json', 'm5/harness/scorecard_m3_texto.csv', 'm5/harness/scorecard.csv', 'm5/harness/scorecard_sin_foto.csv', 'm5/caida/tabla_delta.md', 'm5/caida/entorno.json', 'm5/caida/resultados.json', 'm5/caida/scorecard.csv', 'm5/caida/scorecard_caida.csv']
